# DINOv3 deep fine-tuning

Fresh initialization from the original `model.safetensors`: heads (5 epochs) → last 4 blocks (5 epochs) → fully unfrozen backbone (up to 100 epochs). Checkpoint selection and stage-3 early stopping use **val_unseen Recall@1**.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr')
RUN_NAME = 'dinov3_deeptune_unseen_r1'
RESUME_CHECKPOINT = None  # First run must remain None.
RUN_SMOKE_TEST = True

def find_unique(dataset_slug: str, filename: str) -> Path:
    matches = sorted(
        path for path in INPUT.rglob(filename)
        if path.is_file() and dataset_slug in path.parts
    )
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one {filename} inside {dataset_slug}; found: {matches}'
        )
    return matches[0]

def run_streaming(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=PROJECT, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f'Process failed with exit code {exit_code}. See {log_path}')

full_trainer = find_unique('viscaner-dinov3-code', 'full_finetune.py')
code_root = full_trainer.parent
metadata = find_unique('viscaner-dinov3-data', 'crops_metadata.csv')
manifest = find_unique('viscaner-dinov3-data', 'bottle_images_manifest.csv')
weights = find_unique('viscaner-dinov3-vitb16-weights', 'model.safetensors')
assert metadata.parent == manifest.parent
assert (metadata.parent / 'crops').is_dir()
assert (metadata.parent / 'refs').is_dir()

shutil.copytree(
    code_root, PROJECT, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('dataset-metadata.json'),
)
checksum_path = PROJECT / 'code_sha256.json'
assert checksum_path.is_file(), 'Update viscaner-dinov3-code: code_sha256.json is missing'
checksums = json.loads(checksum_path.read_text(encoding='utf-8'))
for filename, expected in checksums.items():
    candidate = PROJECT / filename
    actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
    assert actual == expected, f'Code integrity check failed: {filename}'

retrieval_source = (PROJECT / 'dinov3_retrieval.py').read_text(encoding='utf-8')
full_source = (PROJECT / 'full_finetune.py').read_text(encoding='utf-8')
required_retrieval_markers = (
    'metadata_for_merge = metadata.drop(columns=["wine_slug"], errors="ignore")',
    'ks: Sequence[int] = (1, 2, 5, 10)',
    'early_stopping_split: str = "val_unseen"',
    'metrics["accuracy"] = metrics["recall_at_1"]',
    'def print_epoch_report(',
)
required_full_markers = (
    'TRAINING_FORMAT = "full-finetune-v2"',
    'validate_files=not quick_smoke',
)
missing_markers = [
    marker for marker in required_retrieval_markers if marker not in retrieval_source
] + [marker for marker in required_full_markers if marker not in full_source]
assert not missing_markers, (
    'Attached viscaner-dinov3-code is outdated. Upload/select its latest version. '
    f'Missing markers: {missing_markers}'
)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5.6,<6', 'safetensors>=0.5,<1',
    'scikit-learn>=1.5,<2', 'PyYAML>=6,<7',
], check=True)

import pandas as pd
import torch
import transformers
import yaml

assert torch.cuda.is_available(), 'Select a GPU accelerator before running this notebook'
assert 'wine_slug' in pd.read_csv(metadata, nrows=1).columns
print('CUDA:', torch.cuda.get_device_name(0), 'GPU count:', torch.cuda.device_count())
print('torch:', torch.__version__, 'transformers:', transformers.__version__)

config_path = PROJECT / 'configs/dinov3_full_finetune.yaml'
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg.update(
    project_root=str(PROJECT), weights_path=str(weights),
    crops_metadata_path=str(metadata), bottle_manifest_path=str(manifest),
    crops_root=str(metadata.parent / 'crops'), refs_root=str(metadata.parent / 'refs'),
    index_path=str(PROJECT / 'datasets/dinov3_retrieval/index.csv'),
    split_summary_path=str(PROJECT / 'datasets/dinov3_retrieval/split_summary.json'),
    models_dir=str(PROJECT / 'models' / RUN_NAME),
    runs_dir=str(PROJECT / 'runs/dinov3_retrieval'),
    run_name=RUN_NAME, device='cuda', amp=True,
)
assert cfg['stage1_epochs'] == 5 and cfg['stage2_epochs'] == 5
assert cfg['stage3_epochs'] == 100 and cfg['unfreeze_last_blocks'] == 4
assert cfg['early_stopping_split'] == 'val_unseen'
assert cfg['early_stopping_metric'] == 'recall_at_1'
if RESUME_CHECKPOINT is None and any(Path(cfg['models_dir']).glob('*.pt')):
    raise RuntimeError('Fresh run found checkpoints. Use a clean session or change RUN_NAME.')
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(json.dumps(cfg, indent=2))


## CUDA smoke test

A disposable model checks all three trainability phases. It skips the expensive full crop inventory; the real run validates the dataset once.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_cfg = dict(cfg)
    smoke_cfg.update(
        models_dir=str(PROJECT / 'smoke/models'),
        runs_dir=str(PROJECT / 'smoke/runs'),
        run_name='deeptune_smoke',
    )
    smoke_config = PROJECT / 'configs/deeptune_smoke.yaml'
    smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
    run_streaming([
        sys.executable, '-u', str(PROJECT / 'train_dinov3_retrieval.py'),
        'all', '--config', str(smoke_config), '--quick-smoke',
    ], PROJECT / 'deeptune_smoke.log')
    print('All three stages passed the disposable CUDA smoke test.')
else:
    print('Smoke test skipped explicitly.')


## Start or continue deep fine-tuning

With `RESUME_CHECKPOINT = None`, initialization comes strictly from the attached safetensors. Epoch output is streamed and also written once to `deeptune_training.log`.

In [ ]:
command = [
    sys.executable, '-u', str(PROJECT / 'train_dinov3_retrieval.py'),
    'all', '--config', str(config_path),
]
if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT)
    assert resume_path.is_file(), f'Resume checkpoint not found: {resume_path}'
    command += ['--resume-checkpoint', str(resume_path)]
print('Starting 5 head + 5 partial + up to 100 full-backbone epochs.', flush=True)
print('Checkpoint and early stopping: val_unseen Recall@1.', flush=True)
run_streaming(command, PROJECT / 'deeptune_training.log')

summary_path = Path(cfg['runs_dir']) / RUN_NAME / 'run_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
required_metrics = {
    'accuracy', 'recall_at_1', 'recall_at_2', 'recall_at_5', 'recall_at_10',
    'mrr', 'median_rank', 'mean_rank', 'num_queries',
}
for split, values in summary['final_metrics'].items():
    missing = required_metrics - set(values)
    assert not missing, f'{split} metrics are incomplete: {sorted(missing)}'
    assert abs(values['accuracy'] - values['recall_at_1']) < 1e-12
print(json.dumps(summary, indent=2))
if summary['continuation_required']:
    print('Session budget reached: save outputs and continue from last.pt.')
else:
    print('Deep fine-tuning completed. Best:', summary['best_checkpoint'])
